# NB19 — Seed replication of the stress test

**Why this notebook exists.** Every stress-test number in the thesis comes from one training run of each model (seed 42). The paired bootstrap in NB16 quantifies the uncertainty of *which articles* were sampled, not the uncertainty of *which training run* produced the model. This notebook retrains both models under a new seed and rescores the same 820 transformed articles, so the effect can be separated from run-to-run variation.

**How to run it.** Set `SEED` in the first cell, then Save Version. Run it once for each seed you want. Seed 42 is already covered by NB10 + NB14 + NB15, so start at 1 and then 2.

**Cost.** One run trains two models at K=9 (about 95 + 75 minutes on a P100) and then scores 820 + 410 articles twice. Budget roughly 3 hours of GPU per seed.

**Output.** `nb19_seed{SEED}_scores.parquet` — one row per generation, carrying both models' probabilities before and after transformation. NB20 aggregates the seeds; nothing is concluded here.

In [ ]:
# ============================ 1/9  configuration ============================
import os, re, json, time, random, pickle, warnings, numpy as np, pandas as pd, torch
from sklearn.exceptions import InconsistentVersionWarning
warnings.filterwarnings('ignore', category=InconsistentVersionWarning)  # the pickled scalers were written under a newer sklearn; only mean_/scale_ are read

SEED = 1                    # <<< EDIT PER RUN: 1, then 2. Seed 42 already exists (NB10/NB14/NB15).

# ---- inputs (Kaggle 'Add input' panel) ----
P_DATASET  = "/kaggle/input/datasets/bahaaqassem/aig-and-humang-dataset/dataset.parquet"
P_GENS     = "/kaggle/input/notebooks/bahaaqassem/nb13-stress-generation-parallel/stress_generations.jsonl"
P_SCALER5  = "/kaggle/input/datasets/bahaaqassem/scalers/scaler.pkl"
P_SCALER11 = "/kaggle/input/datasets/bahaaqassem/scalers/scaler11.pkl"
P_VSTAT16  = "/kaggle/input/datasets/bahaaqassem/aig-16-features/vstat16_scaled.parquet"
P_CHUNKS   = "/kaggle/input/notebooks/bahaaqassem/nb10-trackb-joint-finetune/chunks_K9.npz"  # optional cache

# ---- model and training, identical to NB10/NB14 except SEED ----
MODEL_ID  = "CAMeL-Lab/bert-base-arabic-camelbert-msa"
STAT_COLS = ["burstiness", "ttr", "quote_ratio", "function_word_ratio", "compressibility"]
K_CHUNKS, MAX_CT, STRIDE = 9, 510, 460
HIDDEN, DROPOUT          = 256, 0.0
LR_ENC, LR_HEAD, WD      = 2e-5, 1e-3, 0.01
EPOCHS, MICRO_BS, GRAD_ACCUM = 3, 2, 16
SMOKE = False               # True = 40-article dry run to check plumbing and memory

OUT = f"/kaggle/working/nb19_seed{SEED}_scores.parquet"
DEV = "cuda" if torch.cuda.is_available() else "cpu"

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_everything(SEED)
print(f"[1/9] seed {SEED} | device {DEV} | smoke {SMOKE}", flush=True)
print(f"[1/9] writing to {OUT}", flush=True)

In [ ]:
# ============================ 2/9  data ============================
df = pd.read_parquet(P_DATASET)
if "article_id" in df.columns: df = df.set_index("article_id")
y     = df["label"].to_numpy(np.int64)
split = df["split"].to_numpy()
print(f"[2/9] dataset {len(df)} | human {int((y==0).sum())} | AI {int((y==1).sum())}", flush=True)
for s in ("train", "val", "test"):
    print(f"[2/9]   {s}: {int((split==s).sum())}", flush=True)

rows = [json.loads(l) for l in open(P_GENS, encoding="utf-8")]
G = pd.DataFrame(rows)
G["level"] = G["level"].astype("Float64")
G["gen_id"] = (G.article_id + "__" + G.model + "__" + G.task +
               G.level.map(lambda x: f"__L{int(x)}" if pd.notna(x) else ""))
assert G.gen_id.is_unique, "gen_id not unique"
print(f"[2/9] generations {len(G)} | tasks {G.task.value_counts().to_dict()}", flush=True)
print(f"[2/9] models {G.model.value_counts().to_dict()}", flush=True)

In [ ]:
# ============================ 3/9  the five features ============================
# Pasted verbatim from the notebooks that produced the thesis numbers:
#   NB4  cell 2 : sentence split, whitespace tokens, burstiness, MATTR
#   NB6e cell 2 : diacritic stripping
#   NB6e cell 3 : quote ratio
#   NB6e cell 4 : the 50-item function-word list and its ratio
#   NB6e cell 7 : compressibility
import gzip

_AR_SENT_SPLIT = re.compile(r'[.!?\u061f\u0964\n]+')

def split_sentences(text):
    return [s.strip() for s in _AR_SENT_SPLIT.split(text) if s.strip()]

def tokenize_ws(text):
    return [t for t in re.split(r'\s+', text.strip()) if t]

def burstiness(text):
    L = np.array([len(tokenize_ws(s)) for s in split_sentences(text)], dtype=float)
    if L.size < 2:
        return float('nan')
    mu, sigma = L.mean(), L.std()
    return 0.0 if (sigma + mu) == 0 else float((sigma - mu) / (sigma + mu))

def mattr(tokens, window=100):
    if len(tokens) < window:
        return len(set(tokens)) / len(tokens) if tokens else float('nan')
    ratios = [len(set(tokens[i:i+window])) / window for i in range(len(tokens) - window + 1)]
    return float(np.mean(ratios))

def vocabulary_richness(text, window=100):
    return mattr(tokenize_ws(text), window=window)

AR_DIAC = re.compile(r'[\u064b-\u0652\u0670\u0640]')

def strip_diac(t):
    return AR_DIAC.sub('', t)

QUOTE_PAIRS = [('\u00ab', '\u00bb'), ('\u201c', '\u201d'), ('"', '"'), ("'", "'")]

def quote_ratio(text):
    toks = tokenize_ws(text)
    if not toks: return float('nan')
    inside = 0
    for op, cl in QUOTE_PAIRS:
        if op == cl:
            parts = text.split(op)
            for k in range(1, len(parts), 2):
                inside += len(tokenize_ws(parts[k]))
        else:
            for m in re.finditer(re.escape(op) + r'(.*?)' + re.escape(cl), text, flags=re.S):
                inside += len(tokenize_ws(m.group(1)))
    return float(min(inside, len(toks))) / len(toks)

FUNCTION_WORDS = set(['\u0641\u064a', '\u0645\u0646', '\u0625\u0644\u0649', '\u0639\u0644\u0649',
    '\u0639\u0646', '\u0645\u0639', '\u0628\u064a\u0646', '\u0639\u0646\u062f', '\u0644\u062f\u0649',
    '\u062e\u0644\u0627\u0644', '\u0628\u0639\u062f', '\u0642\u0628\u0644', '\u0645\u0646\u0630',
    '\u062d\u062a\u0649', '\u0625\u0646', '\u0623\u0646', '\u0623\u0646\u0647', '\u0625\u0646\u0647',
    '\u0627\u0644\u0630\u064a', '\u0627\u0644\u062a\u064a', '\u0647\u0630\u0627', '\u0647\u0630\u0647',
    '\u0630\u0644\u0643', '\u062a\u0644\u0643', '\u0647\u0648', '\u0647\u064a', '\u0647\u0645',
    '\u0647\u0646', '\u0646\u062d\u0646', '\u0623\u0646\u0627', '\u0643\u0644', '\u0628\u0639\u0636',
    '\u063a\u064a\u0631', '\u0644\u0627', '\u0645\u0627', '\u0644\u0645', '\u0644\u0646',
    '\u0642\u062f', '\u0644\u0642\u062f', '\u0623\u064a', '\u0623\u064a\u0636\u0627',
    '\u0643\u0645\u0627', '\u0644\u0643\u0646', '\u0623\u0648', '\u062b\u0645', '\u0628\u0644',
    '\u0625\u0630', '\u0625\u0630\u0627', '\u0644\u0623\u0646', '\u062d\u064a\u062b'])

def function_word_ratio(text):
    toks = [strip_diac(t) for t in tokenize_ws(text)]
    if not toks: return float('nan')
    return float(sum(1 for t in toks if t in FUNCTION_WORDS)) / len(toks)

def compressibility(text):
    raw = text.encode('utf-8')
    if len(raw) < 200: return float('nan')
    comp = gzip.compress(raw, compresslevel=6)
    return float(len(comp)) / float(len(raw))

def ttr(text):
    return vocabulary_richness(text)

FEATS = {"burstiness": burstiness, "ttr": ttr, "quote_ratio": quote_ratio,
         "function_word_ratio": function_word_ratio, "compressibility": compressibility}
print("[3/9] feature functions ready:", list(FEATS), flush=True)
print(f"[3/9] function-word list carries {len(FUNCTION_WORDS)} items", flush=True)

In [ ]:
# ============================ 4/9  scalers, reused not refitted ============================
s5  = pickle.load(open(P_SCALER5,  "rb"))
s11 = pickle.load(open(P_SCALER11, "rb"))
print(f"[4/9] scaler.pkl   order={s5['feature_order']}", flush=True)
print(f"[4/9] scaler11.pkl order={s11['feature_order']}", flush=True)

PARAM = {}
for src in (s5, s11):
    order = list(src["feature_order"]); sc = src["scaler"]; med = np.asarray(src["train_median"])
    for f in STAT_COLS:
        if f in order and f not in PARAM:
            i = order.index(f)
            PARAM[f] = (float(sc.mean_[i]), float(sc.scale_[i]), float(med[i]))
missing = [f for f in STAT_COLS if f not in PARAM]
assert not missing, f"no saved scaler parameters for {missing}"
for f in STAT_COLS:
    m, s, md = PARAM[f]; print(f"[4/9]   {f:22s} mean={m:+.4f} scale={s:.4f} median={md:+.4f}", flush=True)

def compute_scaled(texts, tag):
    X = np.zeros((len(texts), len(STAT_COLS)), np.float32); t0 = time.time()
    for i, t in enumerate(texts):
        t = str(t)
        for j, f in enumerate(STAT_COLS):
            v = FEATS[f](t)
            m, s, md = PARAM[f]
            if not np.isfinite(v): v = md
            X[i, j] = (v - m)/s
        if (i+1) % 300 == 0:
            print(f"[4/9]   {tag} features {i+1}/{len(texts)} ({time.time()-t0:.0f}s)", flush=True)
    return X
print("[4/9] scaling routine ready", flush=True)

In [ ]:
# ============================ 5/9  verification against NB6e ============================
v16 = pd.read_parquet(P_VSTAT16)
if "article_id" in v16.columns: v16 = v16.set_index("article_id")
chk = list(dict.fromkeys(G.article_id.tolist()))[:150]
mine   = compute_scaled(df.loc[chk, "text"].tolist(), tag="verify")
theirs = v16.loc[chk, STAT_COLS].to_numpy(np.float32)
ok = True
for j, f in enumerate(STAT_COLS):
    d = np.abs(mine[:, j] - theirs[:, j]); mx = float(d.max())
    r = float(np.corrcoef(mine[:, j], theirs[:, j])[0, 1])
    flag = "OK " if mx < 1e-3 else ("~  " if r > 0.999 else "BAD")
    if flag == "BAD": ok = False
    print(f"[5/9] {flag} {f:22s} max|diff|={mx:.6f} corr={r:.6f}", flush=True)
assert ok, "feature implementation does not reproduce NB6e — stop and investigate"
print("[5/9] feature implementation verified", flush=True)

In [ ]:
# ============================ 6/9  tokenisation, one policy for both models ============================
!pip install -q transformers
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_ID)
CLS, SEP, PAD = tok.cls_token_id, tok.sep_token_id, tok.pad_token_id
CHUNK_LEN = MAX_CT + 2

def chunk_all(texts, tag):
    CH = np.full((len(texts), K_CHUNKS, CHUNK_LEN), PAD, np.int32)
    NC = np.zeros(len(texts), np.int8); t0 = time.time()
    for i, t in enumerate(texts):
        ids = tok(str(t), add_special_tokens=False)["input_ids"] or [tok.unk_token_id]
        wins = [ids[k:k+MAX_CT] for k in range(0, len(ids), STRIDE)][:K_CHUNKS] or [ids[:MAX_CT]]
        NC[i] = len(wins)
        for j, w in enumerate(wins): CH[i, j] = [CLS]+w+[SEP]+[PAD]*(CHUNK_LEN-2-len(w))
        if (i+1) % 500 == 0:
            print(f"[6/9]   {tag} chunked {i+1}/{len(texts)} ({time.time()-t0:.0f}s)", flush=True)
    return CH, NC

if os.path.exists(P_CHUNKS):
    z = np.load(P_CHUNKS); CH, NCH = z["CH"], z["NCH"]
    print(f"[6/9] training chunks loaded from cache {CH.shape}", flush=True)
else:
    CH, NCH = chunk_all(df["text"].tolist(), "train-corpus")
    np.savez_compressed("/kaggle/working/chunks_K9.npz", CH=CH, NCH=NCH)
    print(f"[6/9] training chunks built and cached {CH.shape}", flush=True)

Xstat = compute_scaled(df["text"].tolist(), tag="corpus")
print(f"[6/9] corpus feature matrix {Xstat.shape}", flush=True)

base_ids = list(dict.fromkeys(G.article_id.tolist()))
X_base = compute_scaled(df.loc[base_ids, "text"].tolist(), tag="base")
X_gen  = compute_scaled(G.text.tolist(), tag="gen")
CH_base, NC_base = chunk_all(df.loc[base_ids, "text"].tolist(), "base")
CH_gen,  NC_gen  = chunk_all(G.text.tolist(), "gen")
base_pos = {a: i for i, a in enumerate(base_ids)}
print(f"[6/9] stress matrices | base {CH_base.shape} | gen {CH_gen.shape}", flush=True)

In [ ]:
# ============================ 7/9  model and training, identical to NB10/NB14 ============================
import torch.nn as nn
from transformers import AutoModel, get_linear_schedule_with_warmup
from torch.utils.data import DataLoader, Dataset
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score

class Net(nn.Module):
    def __init__(self, use_stat, stat_dim=5):
        super().__init__()
        self.use_stat = use_stat
        self.enc = AutoModel.from_pretrained(MODEL_ID)
        self.enc.gradient_checkpointing_enable()
        h = self.enc.config.hidden_size
        fused = h + (stat_dim if use_stat else 0)
        self.register_buffer("mu", torch.zeros(fused)); self.register_buffer("sd", torch.ones(fused))
        self.head = nn.Sequential(nn.Linear(fused, HIDDEN), nn.LayerNorm(HIDDEN), nn.ReLU(),
                                  nn.Dropout(DROPOUT), nn.Linear(HIDDEN, 2))
    def encode(self, ids, nch):
        B, K, L = ids.shape; flat = ids.view(B*K, L); att = (flat != PAD).long()
        cls = self.enc(input_ids=flat, attention_mask=att).last_hidden_state[:, 0, :].view(B, K, -1)
        m = (torch.arange(K, device=ids.device)[None, :] < nch[:, None]).float().unsqueeze(-1)
        return (cls*m).sum(1)/m.sum(1).clamp(min=1)
    def forward(self, ids, nch, stat=None):
        v = self.encode(ids, nch)
        if self.use_stat: v = torch.cat([v, stat], 1)
        v = (v - self.mu)/self.sd
        return self.head(v)

class DS(Dataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, k):
        i = self.rows[k]
        return (torch.from_numpy(CH[i].astype(np.int64)), int(NCH[i]),
                torch.from_numpy(Xstat[i]), int(y[i]))
def collate(b):
    return (torch.stack([x[0] for x in b]), torch.tensor([x[1] for x in b]),
            torch.stack([x[2] for x in b]), torch.tensor([x[3] for x in b]))

SMOKE_IDX = set(np.random.RandomState(0).choice(len(df), 40, replace=False)) if SMOKE else set()
train_rows = [i for i in range(len(df)) if split[i] == "train" and (not SMOKE or i in SMOKE_IDX)]
test_rows  = [i for i in range(len(df)) if split[i] == "test"]
print(f"[7/9] train rows {len(train_rows)} | test rows {len(test_rows)}", flush=True)

@torch.no_grad()
def fit_fused_standardizer(model):
    model.eval(); dl = DataLoader(DS(train_rows), batch_size=8, collate_fn=collate)
    n = 0; s = None; s2 = None
    for k, (ids, nch, st, yy) in enumerate(dl):
        ids, nch, st = ids.to(DEV), nch.to(DEV), st.to(DEV)
        v = model.encode(ids, nch)
        if model.use_stat: v = torch.cat([v, st], 1)
        s  = v.sum(0) if s is None else s + v.sum(0)
        s2 = (v*v).sum(0) if s2 is None else s2 + (v*v).sum(0)
        n += v.shape[0]
        if (k+1) % 100 == 0: print(f"[7/9]   standardizer {n} articles", flush=True)
    mu = s/n; sd = torch.sqrt(torch.clamp(s2/n - mu*mu, min=1e-6))
    model.mu.copy_(mu); model.sd.copy_(sd)

def train_one(use_stat, tag):
    seed_everything(SEED)
    model = Net(use_stat).to(DEV)
    fit_fused_standardizer(model)
    enc_p  = [p for n, p in model.named_parameters() if n.startswith("enc.")]
    head_p = [p for n, p in model.named_parameters() if not n.startswith("enc.")]
    opt = torch.optim.AdamW([{"params": enc_p, "lr": LR_ENC},
                             {"params": head_p, "lr": LR_HEAD}], weight_decay=WD)
    cw = compute_class_weight("balanced", classes=np.array([0, 1]), y=y[train_rows])
    lossf = nn.CrossEntropyLoss(weight=torch.tensor(cw, dtype=torch.float32, device=DEV))
    dl = DataLoader(DS(train_rows), batch_size=MICRO_BS, shuffle=True, collate_fn=collate, drop_last=True)
    total = (len(dl)//GRAD_ACCUM)*EPOCHS
    sched = get_linear_schedule_with_warmup(opt, int(0.06*total), total)
    scaler = torch.amp.GradScaler('cuda')
    model.train(); t0 = time.time(); step = 0
    for ep in range(EPOCHS):
        opt.zero_grad()
        for bi, (ids, nch, st, yy) in enumerate(dl):
            ids, nch, st, yy = ids.to(DEV), nch.to(DEV), st.to(DEV), yy.to(DEV)
            with torch.amp.autocast('cuda'):
                logits = model(ids, nch, st); loss = lossf(logits, yy)/GRAD_ACCUM
            scaler.scale(loss).backward()
            if (bi+1) % GRAD_ACCUM == 0:
                scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(); step += 1
                if step % 20 == 0:
                    print(f"[7/9] {tag} seed{SEED} ep{ep} step {step}/{total} "
                          f"loss {float(loss)*GRAD_ACCUM:.4f} ({time.time()-t0:.0f}s)", flush=True)
        print(f"[7/9] {tag} seed{SEED} epoch {ep} done ({time.time()-t0:.0f}s)", flush=True)
    torch.save({"model": model.state_dict(), "seed": SEED, "use_stat": use_stat},
               f"/kaggle/working/nb19_{tag}_seed{SEED}.pt")
    print(f"[7/9] {tag} seed{SEED} trained in {(time.time()-t0)/60:.1f} min", flush=True)
    return model

@torch.no_grad()
def predict(model, CHm, NCm, Xm):
    model.eval(); P = []
    for i in range(0, len(CHm), 8):
        ids = torch.from_numpy(CHm[i:i+8].astype(np.int64)).to(DEV)
        nch = torch.tensor(NCm[i:i+8].astype(np.int64)).to(DEV)
        st  = torch.from_numpy(Xm[i:i+8]).to(DEV)
        with torch.amp.autocast('cuda'):
            p = torch.softmax(model(ids, nch, st), 1)[:, 1]
        P.append(p.float().cpu().numpy())
    return np.concatenate(P)
print("[7/9] model, training and prediction routines ready", flush=True)

In [ ]:
# ============================ 8/9  train both models, score the stress set ============================
PRED = {}
for use_stat, tag in ((True, "hybrid"), (False, "neural")):
    model = train_one(use_stat, tag)
    Xte = Xstat[test_rows]
    p_te = predict(model, CH[test_rows], NCH[test_rows], Xte)
    yt = y[test_rows]; pred = (p_te >= 0.5).astype(int)
    print(f"[8/9] {tag} seed{SEED} STANDARD SPLIT  MacroF1 {100*f1_score(yt,pred,average='macro'):.2f}  "
          f"Acc {100*accuracy_score(yt,pred):.2f}  AUC {100*roc_auc_score(yt,p_te):.2f}", flush=True)
    PRED[(tag, "base")] = predict(model, CH_base, NC_base, X_base)
    PRED[(tag, "gen")]  = predict(model, CH_gen,  NC_gen,  X_gen)
    print(f"[8/9] {tag} seed{SEED} scored {len(PRED[(tag,'base')])} originals "
          f"and {len(PRED[(tag,'gen')])} transformed articles", flush=True)
    del model; torch.cuda.empty_cache()

In [ ]:
# ============================ 9/9  write one row per generation ============================
G = G.reset_index(drop=True)
bidx = np.array([base_pos[a] for a in G.article_id])
out = G[["gen_id", "article_id", "model", "task", "level"]].copy()
out["seed"] = SEED
for tag in ("hybrid", "neural"):
    out[f"p_{tag}_base"] = PRED[(tag, "base")][bidx]
    out[f"p_{tag}_gen"]  = PRED[(tag, "gen")]
out.to_parquet(OUT, index=False)
print(f"[9/9] wrote {OUT} | rows {len(out)}", flush=True)

def rate(p): return 100.0*float((np.asarray(p) >= 0.5).mean()) if len(p) else float('nan')
pol = out[out.task == "polish"]; hum = out[out.task == "humanize"]
print("[9/9] quick read, before the validity gate is applied in NB20:", flush=True)
print(f"[9/9]   polish   n={len(pol)}  FPR hybrid {rate(pol.p_hybrid_gen):.2f}  neural {rate(pol.p_neural_gen):.2f}", flush=True)
print(f"[9/9]   humanize n={len(hum)}  TPR hybrid {rate(hum.p_hybrid_gen):.2f}  neural {rate(hum.p_neural_gen):.2f}", flush=True)
print(f"[9/9]   control  polish base FPR hybrid {rate(pol.p_hybrid_base):.2f}  neural {rate(pol.p_neural_base):.2f}", flush=True)
print("[9/9] done. Change SEED and Save Version again for the next seed.", flush=True)